In [ ]:
# Data Manipulation
import pandas as pd
import numpy as np

# Preprocessing
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

# Machine Learning Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
# from xgboost import XGBClassifier

# Model Evaluation
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, roc_auc_score, roc_curve

# Dimensionality Reduction (if needed)
from sklearn.decomposition import PCA

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# Warnings
import warnings
warnings.filterwarnings('ignore')


In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

# Load the dataset
df = pd.read_csv("sentinel_features_for_modeling.csv")

# Drop the 'nameOrig' and 'nameDest' columns, which are identifiers for sender and receiver.
# These columns are not useful for modeling because they don't carry predictive information and may add noise.
df = df.drop(['nameOrig', 'nameDest'], axis=1)

# Convert the 'type' categorical column into dummy/one-hot encoded variables.
# 'drop_first=True' avoids multicollinearity by dropping the first category (acts as a baseline).
df = pd.get_dummies(df, columns=['type'], drop_first=True)

# Ensure the 'balanced' column is stored as integers (0 or 1) instead of boolean or float.
# This is important for consistent handling in modeling tasks like classification.
df['balanced'] = df['balanced'].astype(int)

# Drop columns related to raw transaction amounts and balances.
# These columns may leak information or be highly correlated, which can hurt model performance.
df = df.drop(['amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest'], axis=1)

# Separate features (X) and target variable (y).
# 'isFraud' is the label we want to predict, so it's removed from the features.
X = df.drop("isFraud", axis=1)
y = df["isFraud"]

# Double-check for any remaining categorical columns and encode them (safety step)
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
if categorical_cols:
    X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

# Apply standard scaling to the feature set.
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Optional: Convert to DataFrame for readability
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)

# Display
print("Final preprocessed shape:", X_scaled_df.shape)
print(X_scaled_df.head())
print("Target distribution:\n", y.value_counts(normalize=True))


In [ ]:
df['balanced'] = df['balanced'].astype(int)

In [ ]:
# Split the data into training and testing sets.
# Use stratified sampling to preserve the original fraud/non-fraud ratio in both sets.
# 20% of the data is reserved for testing; random_state ensures reproducibility.
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

X_train = pd.DataFrame(X_train)
X_test = pd.DataFrame(X_test)
y_train = pd.DataFrame(y_train)
y_test = pd.DataFrame(y_test)

# Save the training and testing sets to CSV files.
# This allows for easy reuse later without needing to repeat preprocessing steps.
X_train.to_csv("X_train.csv", index=False)
X_test.to_csv("X_test.csv", index=False)
y_train.to_csv("y_train.csv", index=False)
y_test.to_csv("y_test.csv", index=False)

# Print the shapes of the training and test feature sets to confirm the split.
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

# Check the class distribution in the training and test labels.
# Ensures stratified sampling preserved the fraud class balance across sets.
print("y_train distribution:\n", y_train.value_counts(normalize=True))
print("y_test distribution:\n", y_test.value_counts(normalize=True))


In [ ]:
# Display the count of each class in the target variable `y`.
# This helps us understand how imbalanced the dataset is (e.g., number of fraud vs non-fraud cases).
print("Class distribution in full dataset (y):")
print(y.value_counts())

# Display class distribution as percentages for better interpretation of imbalance.
print("\nClass balance (percentages):")
print(y.value_counts(normalize=True).apply(lambda x: f"{x:.2%}"))

# Import seaborn and matplotlib for visualizations.
import seaborn as sns
import matplotlib.pyplot as plt

# Set seaborn style for cleaner plots.
sns.set(style="whitegrid")

# Create a bar plot to visualize the distribution of fraud vs. non-fraud cases.
# This helps visualize class imbalance.
plt.figure(figsize=(6,4))
sns.countplot(x=y)
plt.title("Fraud vs Not Fraud Distribution")
plt.xlabel("isFraud")
plt.ylabel("Count")
plt.show()

# Create a heatmap to visualize correlation between features.
# This helps identify redundant features or strong linear relationships.
plt.figure(figsize=(10,8))
sns.heatmap(X.corr(), cmap="coolwarm", annot=False, linewidths=0.5)
plt.title("Feature Correlation Heatmap")
plt.show()


In [ ]:
print("\nOriginal features shape:", X.shape)
print("Scaled features shape:", X_scaled.shape)
print("Scaled sample:", X_scaled[:2])
print("Target distribution:\n", y.value_counts(normalize=True))

